# Lab 8 - dataframes con spark

## Sebastian Juarez - 21471
## Juan Pablo Cordón - 21458

## Avances

### Toma de datos y fusión

In [1]:
import pyreadstat

In [2]:
# Spark y utilidades
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

import pandas as pd
import zipfile, io, re, os, unicodedata
from pathlib import Path

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("AccidentesGT-EDA").getOrCreate()

ZIP_PATH   = "./Accidentes.zip"
STAGING_DIR = "./accidentes_staging"
Path(STAGING_DIR).mkdir(parents=True, exist_ok=True)

def normalize_col(col: str) -> str:
    s = unicodedata.normalize('NFKD', str(col))
    s = "".join([c for c in s if not unicodedata.combining(c)])
    s = re.sub(r"[^A-Za-z0-9_]+", "_", s.strip().lower())
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def normalize_df_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_col(c) for c in df.columns]
    return df

def read_sav_bytes(b: bytes) -> pd.DataFrame:
    bio = io.BytesIO(b)
    try:
        pdf = pd.read_spss(bio, convert_categoricals=False)
        return pdf
    except Exception:
        pass
    try:
        pdf, meta = pyreadstat.read_sav(io.BytesIO(b))
        return pdf
    except Exception as e:
        raise RuntimeError("No se pudo leer el .sav; instala 'pyreadstat'") from e


In [ ]:
# Ruta temporal para los sav (error con windows)

import os, io, re, unicodedata, zipfile, tempfile, pathlib
import pandas as pd

try:
    import pyreadstat
    print("pyreadstat OK - versión:", pyreadstat.__version__)
except Exception as e:
    print("pyreadstat no se pudo importar en este kernel:", repr(e))

def read_sav_bytes(b: bytes) -> pd.DataFrame:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".sav") as tmp:
        tmp.write(b)
        tmp_path = tmp.name

    try:
        try:
            pdf = pd.read_spss(tmp_path, convert_categoricals=False)
            return pdf
        except Exception as e1:
            try:
                import pyreadstat
                pdf, meta = pyreadstat.read_sav(tmp_path, apply_value_formats=False, formats_as_category=False)
                return pdf
            except Exception as e2:
                raise RuntimeError(f"Fallo leyendo .sav; pandas_error={repr(e1)} | pyreadstat_error={repr(e2)}")
    finally:
        try:
            os.unlink(tmp_path)
        except Exception:
            pass

print("Hotfix read_sav_bytes cargado.")


pyreadstat OK - versión: 1.2.7
Hotfix read_sav_bytes cargado.


In [ ]:
import numpy as np
import pandas as pd

NUM_REGEX = r"^[+-]?(\d+([.,]\d+)?|\.\d+)$"

def smart_cast_dataframe(df: pd.DataFrame, numeric_threshold: float = 0.95) -> pd.DataFrame:
    df = df.copy()

    NULL_LIKE = {"", "nan", "none", "sin dato", "s/d", "na", "n/a", "null"}
    for c in df.columns:
        if df[c].dtype == object:
            s = df[c].astype(str).str.strip()
            s = s.where(~s.str.lower().isin(NULL_LIKE), other=pd.NA)
            df[c] = s

    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            s = df[c]
            s_dot = s.str.replace(",", ".", regex=False)
            mask_num = s_dot.fillna("").str.match(NUM_REGEX)
            denom = (s_dot.notna()).sum()
            ratio = (mask_num.sum() / denom) if denom else 0.0

            if ratio >= numeric_threshold and denom > 0:
                df[c] = pd.to_numeric(s_dot, errors="coerce")
            else:
                try:
                    df[c] = s.astype("string[pyarrow]")
                except Exception:
                    df[c] = s.astype("string")

    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            s = df[c].dropna()
            if len(s) and np.all(np.isclose(s, np.round(s))):
                df[c] = df[c].round().astype("Int64")

    try:
        df = df.convert_dtypes(dtype_backend="pyarrow")
    except Exception:
        pass

    return df


In [ ]:
BUCKETS = {
    "hechos":      re.compile(r"hecho|incidente|accidente|hechos", re.I),
    "vehiculos":   re.compile(r"vehicul", re.I),
    "fallecidos":  re.compile(r"fallecid|muert", re.I),
}

dfs_acc = {k: [] for k in BUCKETS.keys()}
bucket_files = {k: [] for k in BUCKETS.keys()}

# Extrae el zip

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    members = z.namelist()
    data_files = [n for n in members if n.lower().endswith((".sav", ".sas7bdat", ".xls", ".xlsx"))]
    for fname in data_files:
        bkt = None
        base = Path(fname).name
        for k, rx in BUCKETS.items():
            if rx.search(base):
                bkt = k
                break
        if bkt is None:
            continue

        with z.open(fname) as f:
            raw = f.read()

        ext = base.lower().split(".")[-1]
        try:
            if ext == "sav":
                pdf = read_sav_bytes(raw)
            elif ext in ("xls", "xlsx"):
                xls = pd.ExcelFile(io.BytesIO(raw))
                for sh in xls.sheet_names:
                    try:
                        tmp = pd.read_excel(io.BytesIO(raw), sheet_name=sh, dtype=str)
                        if tmp.shape[0] == 0:
                            continue
                        tmp = normalize_df_cols(tmp)
                        dfs_acc[bkt].append(tmp)
                        bucket_files[bkt].append(f"{base}::{sh}")
                    except Exception:
                        pass
                continue
            else:
                continue

            if isinstance(pdf, pd.DataFrame) and pdf.shape[0] > 0:
                pdf = normalize_df_cols(pdf)
                dfs_acc[bkt].append(pdf)
                bucket_files[bkt].append(base)
        except Exception as e:
            print(f"No se pudo leer {base}: {e}")

for k in dfs_acc.keys():
    print(f"\n=== {k.upper()} ===")
    for it in bucket_files[k]:
        print(" -", it)

# Combina por bucket y guarda como parquet
combined_paths = {}
for k, lst in dfs_acc.items():
    if not lst:
        continue
    combo = pd.concat(lst, ignore_index=True)
    combo = combo.dropna(axis=1, how="all")

    combo = smart_cast_dataframe(combo, numeric_threshold=0.95)

    out_path = f"{STAGING_DIR}/{k}_pandas.parquet"
    combo.to_parquet(out_path, index=False)
    combined_paths[k] = out_path

combined_paths



=== HECHOS ===
 - 2013 hechos.sav
 - 2014 hechos.sav
 - 2015 hechos.xlsx::Eventos de tránsito
 - 2016 hechos.sav
 - 2017 hechos.sav
 - 2018 hechos.sav
 - 2019 hechos.sav
 - 2020 hechos.sav
 - 2021 hechos.sav
 - 2022 hechos.sav
 - 2023 hechos.sav

=== VEHICULOS ===
 - 2013 vehiculos.sav
 - 2014 vehiculos.sav
 - 2015 vehiculos.sav
 - 2015 vehiculos.xlsx::Vehículos involucrados
 - 2017 vehiculos.sav
 - 2018 vehiculos.sav
 - 2019 vehiculos.sav
 - 2020 vehiculos.sav
 - 2021 vehiculos.sav
 - 2022 vehiculos.sav
 - 2023 vehiculos.sav

=== FALLECIDOS ===
 - 2013 fallecidos.sav
 - 2014 fallecidos.sav
 - 2015 fallecidos.xlsx::Fallecidos y Lesionados
 - 2016 fallecidos.sav
 - 2017 fallecidos.sav
 - 2018 fallecidos.sav
 - 2019 fallecidos.sav
 - 2020 fallecidos.sav
 - 2021 fallecidos.sav
 - 2022 fallecidos.sav
 - 2023 fallecidos.sav


{'hechos': './accidentes_staging/hechos_pandas.parquet',
 'vehiculos': './accidentes_staging/vehiculos_pandas.parquet',
 'fallecidos': './accidentes_staging/fallecidos_pandas.parquet'}

In [ ]:
sdf = {}
for name, path in combined_paths.items():
    df = spark.read.parquet(path)
    for c in df.columns:
        df = df.withColumnRenamed(c, normalize_col(c))
    sdf[name] = df.cache()

{ k: (v.count(), len(v.columns)) for k, v in sdf.items() }


{'hechos': (76465, 38), 'vehiculos': (112030, 38), 'fallecidos': (108941, 41)}

### Estandarizacion de nombres

In [ ]:
from pyspark.sql import functions as F

ALIASES = {
    "anio": [
        "anio", "ano", "year", "ano_",
        "anio_ocu", "ano_ocu", "year_ocu"
    ],
    "mes": [
        "mes", "mes_num", "mesnumero", "mes_", "mes_nombre",
        "mes_ocu"
    ],
    "departamento": [
        "departamento", "depto", "depto_", "departament", "departamento_",
        "depto_ocu", "departamento_ocu"
    ],
    "tipo_accidente": [
        "tipo_accidente", "tipo", "clase_accidente",
        "tipo_hecho", "clase_hecho", "evento", "tipo_evento",
        "clase_acc", "causa_acc"
    ],
}

MES_MAP = {
    "ene":1,"enero":1,"jan":1,"january":1,
    "feb":2,"febrero":2,"february":2,
    "mar":3,"marzo":3,"march":3,
    "abr":4,"abril":4,"apr":4,"april":4,
    "may":5,"mayo":5,
    "jun":6,"junio":6,
    "jul":7,"julio":7,
    "ago":8,"agosto":8,"aug":8,"august":8,
    "sep":9,"set":9,"sept":9,"septiembre":9,"september":9,
    "oct":10,"octubre":10,"oct":10,"october":10,
    "nov":11,"noviembre":11,"november":11,
    "dic":12,"diciembre":12,"dec":12,"december":12,
}

def coalesce_col(df, target, candidates):
    cols = [c for c in candidates if c in df.columns]
    if not cols:
        return df, None
    expr = None
    for c in cols:
        expr = F.coalesce(expr, F.col(c)) if expr is not None else F.col(c)
    return df.withColumn(target, expr), target

def ensure_keys(df):
    for target, cands in ALIASES.items():
        df, _ = coalesce_col(df, target, cands)

    if "anio" in df.columns:
        df = df.withColumn("anio", F.col("anio").cast("int"))

    if "mes" in df.columns:
        df = df.withColumn("mes", F.col("mes").cast("int"))
        mes_expr = F.lower(F.regexp_replace(F.coalesce(F.col("mes").cast("string"), F.lit("")), r"[^a-zA-Z]+", ""))
        mes_map_lit = F.create_map([F.lit(k) for kv in MES_MAP.items() for k in kv])
        df = df.withColumn(
            "mes",
            F.when(F.col("mes").isNull(), mes_map_lit.getItem(mes_expr))
             .otherwise(F.col("mes")).cast("int")
        )

    if "departamento" in df.columns:
        df = df.withColumn("departamento", F.trim(F.initcap(F.col("departamento"))))

    if "tipo_accidente" in df.columns:
        clean = F.regexp_replace(F.lower(F.col("tipo_accidente")), r"\s+", " ")
        df = df.withColumn("tipo_accidente", F.trim(clean))
        df = (df
            .withColumn("tipo_accidente",
                F.when(F.col("tipo_accidente").isin("colision","colisión"), F.lit("colision"))
                 .when(F.col("tipo_accidente").isin("atropello","atropellamiento"), F.lit("atropello"))
                 .otherwise(F.col("tipo_accidente"))
            )
        )

    return df

for k in list(sdf.keys()):
    sdf[k] = ensure_keys(sdf[k]).cache()

for k, df in sdf.items():
    have = {c for c in ["anio","mes","departamento","tipo_accidente"] if c in df.columns}
    print(f"{k}: claves presentes -> {sorted(have)}")


C:\Users\sebas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\pyspark\sql\classic\column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


hechos: claves presentes -> ['anio', 'departamento', 'mes', 'tipo_accidente']
vehiculos: claves presentes -> ['anio', 'departamento', 'mes', 'tipo_accidente']
fallecidos: claves presentes -> ['anio', 'departamento', 'mes', 'tipo_accidente']


### Conteo de datos

In [ ]:
for k, df in sdf.items():
    print(f"=== {k.upper()} ===")
    print("Filas:", df.count(), " | Columnas:", len(df.columns))
    df.show(10, truncate=False)

DESCRIBE_COLS = {
    "hechos": ["anio", "mes", "departamento", "tipo_accidente"],
    "vehiculos": ["anio", "mes", "departamento", "tipo_accidente"],
    "fallecidos": ["anio", "mes", "departamento", "tipo_accidente"],
    "lesionados": ["anio", "mes", "departamento", "tipo_accidente"],
}

for k, df in sdf.items():
    print(f"\n--- Describe: {k} ---")
    cols = [c for c in DESCRIBE_COLS.get(k, []) if c in df.columns]
    if cols:
        df.select(*cols).describe().show(truncate=False)
        num_cols = [c for c, t in df.dtypes if t in ("int", "bigint", "double", "float")]
        if num_cols:
            df.select(*num_cols).summary().show(truncate=False)


=== HECHOS ===
Filas: 76465  | Columnas: 42
+---------+-------+-------+-----------+--------+------+---------+---------+---------+--------+--------+--------+--------+-----------+--------+---------+----------+---------+---------+----------+---------------+----------+------------+--------+--------+------+----------+--------+---------+-------+--------+--------+--------+-------------+-------------+-----------------+------------+-----------+----+---+------------+--------------+
|num_hecho|dia_ocu|mes_ocu|dia_sem_ocu|hora_ocu|g_hora|depto_ocu|mupio_ocu|areag_ocu|zona_ocu|sexo_pil|edad_pil|g_edad_2|mayor_menor|tipo_veh|color_veh|modelo_veh|causa_acc|marca_veh|estado_pil|num_correlativo|corre_base|area_geo_ocu|sexo_con|edad_con|g_edad|estado_con|tipo_eve|num_corre|ano_ocu|g_hora_5|sexo_per|edad_per|g_edad_80ymas|g_edad_60ymas|edad_quinquenales|g_modelo_veh|zona_ciudad|anio|mes|departamento|tipo_accidente|
+---------+-------+-------+-----------+--------+------+---------+---------+---------+-----

### Años y verificación

In [ ]:
from pyspark.sql import functions as F

years_by_ds = {}
nulls_by_ds = {}

for k, df in sdf.items():
    if "anio" in df.columns:
        ys = [r["anio"] for r in df.select("anio")
                              .where(F.col("anio").isNotNull())
                              .distinct().orderBy("anio").collect()]
        years_by_ds[k] = ys

        nulls_by_ds[k] = df.filter(F.col("anio").isNull()).count()
        print(f"{k}: {ys} (filas con anio nulo = {nulls_by_ds[k]})")
    else:
        print(f"{k}: (sin columna 'anio')")

if years_by_ds:
    sets = list(map(set, years_by_ds.values()))
    inter = set.intersection(*sets) if len(sets) > 1 else sets[0]
    union = set.union(*sets)
    print("\nIntersección de años:", sorted(inter))
    print("Unión de años:", sorted(union))
    if all(set(ys) == inter for ys in years_by_ds.values()):
        print("✔ Todos los datasets tienen exactamente los mismos años (ignorando nulos).")
    else:
        print("⚠ Hay diferencias de años entre datasets (ignorando nulos).")


hechos: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022] (filas con anio nulo = 11975)
vehiculos: [2016, 2017, 2018, 2019, 2020, 2021, 2022] (filas con anio nulo = 24050)
fallecidos: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022] (filas con anio nulo = 18050)

Intersección de años: [2016, 2017, 2018, 2019, 2020, 2021, 2022]
Unión de años: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
⚠ Hay diferencias de años entre datasets (ignorando nulos).


### Valores distintos por accidente

In [14]:
for k, df in sdf.items():
    if "tipo_accidente" in df.columns:
        print(f"\n{k}: tipos de accidente (distinct)")
        df.select("tipo_accidente").distinct().orderBy("tipo_accidente").show(200, truncate=False)



hechos: tipos de accidente (distinct)
+--------------+
|tipo_accidente|
+--------------+
|NULL          |
|1             |
|2             |
|3             |
|4             |
|5             |
|99            |
+--------------+


vehiculos: tipos de accidente (distinct)
+--------------+
|tipo_accidente|
+--------------+
|NULL          |
|1             |
|2             |
|3             |
|4             |
|99            |
+--------------+


fallecidos: tipos de accidente (distinct)
+--------------+
|tipo_accidente|
+--------------+
|NULL          |
|1             |
|2             |
|3             |
|4             |
|5             |
|99            |
+--------------+



### Departamentos unicos por base de datos

In [ ]:
for k, df in sdf.items():
    if "departamento" in df.columns:
        cnt = df.select("departamento").distinct().count()
        print(f"{k}: departamentos únicos = {cnt}")
        df.select("departamento").distinct().orderBy("departamento").show(50, truncate=False)


hechos: departamentos únicos = 22
+------------+
|departamento|
+------------+
|1           |
|10          |
|11          |
|12          |
|13          |
|14          |
|15          |
|16          |
|17          |
|18          |
|19          |
|2           |
|20          |
|21          |
|22          |
|3           |
|4           |
|5           |
|6           |
|7           |
|8           |
|9           |
+------------+

vehiculos: departamentos únicos = 22
+------------+
|departamento|
+------------+
|1           |
|10          |
|11          |
|12          |
|13          |
|14          |
|15          |
|16          |
|17          |
|18          |
|19          |
|2           |
|20          |
|21          |
|22          |
|3           |
|4           |
|5           |
|6           |
|7           |
|8           |
|9           |
+------------+

fallecidos: departamentos únicos = 22
+------------+
|departamento|
+------------+
|1           |
|10          |
|11          |
|12          |
|13 